# Fine-tune the small claim verifier (LoRA, free T4)

Trains the BioResearch Agent's claim verifier: **Qwen/Qwen3-1.7B** (Apache-2.0) with a 3-way
classification head (SUPPORTED / CONTRADICTED / NOT_ENOUGH_INFO) and LoRA adapters, in 4-bit
(QLoRA) so it fits a free Colab or Kaggle T4.

**Before you start:** get `verifier_bundle.zip` (made by `uv run python
scripts/make_verifier_data.py bundle`, at `data/finetune/verifier_bundle.zip`). It holds the
teacher-labelled train/val/test JSONL and a README with these settings.

1. **GPU:** Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator >
   GPU T4 (and Internet on, to download the model).
2. **Data, one of:** upload the zip when the next cells ask (Colab); or put it in Google Drive
   at `MyDrive/verifier_bundle.zip` and run the Drive cell; or add it to Kaggle as a dataset
   (Add data > Upload).
3. **Run all.** It prints the steps, an ETA and validation metrics, and at the end saves
   `verifier-lora.zip` (adapter + classification head + tokenizer + `training.json`).
4. **Back in the repo:** unzip it into `models/verifier-lora/` (so that
   `models/verifier-lora/training.json` exists), then run
   `uv run python scripts/eval_verifier.py`; use it with `VERIFIER_BACKEND=small_lora` or
   `router`.

In [ ]:
!pip -q install "transformers>=5.0" "peft>=0.17" "accelerate>=1.0" "bitsandbytes>=0.45"

Optional (Colab + Google Drive): mount Drive if you put the bundle at
`MyDrive/verifier_bundle.zip`. Skip this cell if you upload the file or use Kaggle.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
import glob
import os
import zipfile
from pathlib import Path

candidates = [
    "verifier_bundle.zip",
    "/content/drive/MyDrive/verifier_bundle.zip",
    *glob.glob("/kaggle/input/**/verifier_bundle.zip", recursive=True),
]
bundle = next((p for p in candidates if os.path.exists(p)), None)
if bundle is None:
    try:
        from google.colab import files  # Colab: ask for the file

        print("Choose verifier_bundle.zip to upload")
        bundle = next(iter(files.upload()))
    except ImportError as exc:
        raise SystemExit(
            "verifier_bundle.zip not found: add it as a Kaggle dataset or copy it into the "
            "working directory"
        ) from exc
DATA = Path("bundle")
zipfile.ZipFile(bundle).extractall(DATA)
print(f"using {bundle}\n")
print((DATA / "README.txt").read_text())

In [ ]:
import json
import random
import time

import torch
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen3-1.7B"
LABELS = ["SUPPORTED", "CONTRADICTED", "NOT_ENOUGH_INFO"]  # same order as bioagent.finetune
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
STEPS, BATCH, ACCUM, LR, MAX_LEN, WARMUP = 600, 8, 2, 2e-4, 512, 20
EVAL_EVERY, LOG_EVERY = 100, 25
assert torch.cuda.is_available(), "no GPU: switch the runtime to a T4"
print(torch.cuda.get_device_name(0))


# same as bioagent.finetune.model.format_input
def format_input(claim, passage):
    return f"Claim: {claim}\nPassage: {passage}\nDoes the passage support the claim?"


def load(path):
    rows = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    return [r for r in rows if (r.get("labels") or {}).get("teacher") in LABEL2ID]


train, val = load(DATA / "train.jsonl"), load(DATA / "val.jsonl")
print(len(train), "train /", len(val), "val pairs")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=3,
    quantization_config=bnb,
    device_map={"": 0},
    id2label=dict(enumerate(LABELS)),
    label2id=LABEL2ID,
)
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        modules_to_save=["score"],
    ),
)
model.print_trainable_parameters()

In [ ]:
counts = [sum(r["labels"]["teacher"] == lab for r in train) for lab in LABELS]
weights = torch.tensor([len(train) / (3 * max(1, c)) for c in counts], device="cuda")
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1, (s + 1) / WARMUP) * max(0.05, 1 - s / STEPS)
)


def batch(rows):
    enc = tok(
        [format_input(r["claim"], r["passage"]) for r in rows],
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    return enc, torch.tensor([LABEL2ID[r["labels"]["teacher"]] for r in rows], device="cuda")


def predict(rows):
    model.eval()
    preds = []
    with torch.inference_mode():
        for i in range(0, len(rows), 16):
            enc, _ = batch(rows[i : i + 16])
            preds += model(**enc).logits.argmax(-1).tolist()
    model.train()
    return [LABELS[i] for i in preds]


def metrics(rows):
    gold, pred = [r["labels"]["teacher"] for r in rows], predict(rows)
    pairs = list(zip(gold, pred, strict=True))
    f1 = {}
    for lab in LABELS:
        tp = sum(g == lab and p == lab for g, p in pairs)
        fp = sum(g != lab and p == lab for g, p in pairs)
        fn = sum(g == lab and p != lab for g, p in pairs)
        prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn)
        f1[lab] = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    present = [lab for lab in LABELS if lab in gold]
    return {
        "accuracy": sum(g == p for g, p in pairs) / max(1, len(gold)),
        "macro_f1": sum(f1[lab] for lab in present) / max(1, len(present)),
        "per_class_f1": f1,
        "confusion": {
            g: {p: sum(1 for a, b in pairs if a == g and b == p) for p in LABELS} for g in LABELS
        },
    }


rng, t0, history = random.Random(0), time.time(), []
model.train()
print(f"training {STEPS} steps x {BATCH * ACCUM} examples")
for step in range(1, STEPS + 1):
    opt.zero_grad()
    for _ in range(ACCUM):
        enc, y = batch(rng.sample(train, BATCH))
        (loss_fn(model(**enc).logits.float(), y) / ACCUM).backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    opt.step()
    sched.step()
    if step % LOG_EVERY == 0 or step == 1:
        per_step = (time.time() - t0) / step
        eta = per_step * (STEPS - step) / 60
        print(
            f"step {step}/{STEPS}  {per_step:.2f}s/step  ETA {eta:.0f} min  "
            f"GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
        )
    if step % EVAL_EVERY == 0:
        m = metrics(val)
        history.append({"step": step, "accuracy": m["accuracy"], "macro_f1": m["macro_f1"]})
        print(f"  val accuracy {m['accuracy']:.3f}  macro-F1 {m['macro_f1']:.3f}")
train_seconds = time.time() - t0

In [ ]:
final = metrics(val)
print(f"final val accuracy {final['accuracy']:.3f}  macro-F1 {final['macro_f1']:.3f}")
print("per-class F1", {k: round(v, 3) for k, v in final["per_class_f1"].items()})
print("confusion (rows = teacher, columns = model):")
for g, row in final["confusion"].items():
    print(f"  {g:<16}", row)

out = Path("verifier-lora")
out.mkdir(exist_ok=True)
model.save_pretrained(out)  # LoRA adapter + the classification head (modules_to_save)
tok.save_pretrained(out)
meta = {
    "base_model": BASE,
    "label_source": "teacher",
    "steps": STEPS,
    "batch": BATCH,
    "accum": ACCUM,
    "effective_batch": BATCH * ACCUM,
    "lr": LR,
    "max_length": MAX_LEN,
    "quantization": "4-bit nf4 (QLoRA)",
    "device": torch.cuda.get_device_name(0),
    "train_pairs": len(train),
    "train_seconds": round(train_seconds, 1),
    "seconds_per_step": round(train_seconds / STEPS, 2),
    "peak_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "params": {
        "total": sum(p.numel() for p in model.parameters()),
        "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad),
    },
    "val": final,
    "history": history,
}
(out / "training.json").write_text(json.dumps(meta, indent=1))
zip_path = "verifier-lora.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in out.rglob("*"):
        zf.write(f, f.relative_to(out.parent))
print(f"saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
try:
    from google.colab import files

    files.download(zip_path)  # Colab: browser download
except ImportError:
    print("Kaggle: download it from the Output panel (/kaggle/working/verifier-lora.zip)")

## Back in the repo

```bash
unzip verifier-lora.zip -d models/      # creates models/verifier-lora/training.json etc.
uv run python scripts/eval_verifier.py  # test split: small vs DeBERTa vs Gemini; router curve
VERIFIER_BACKEND=router uv run bioagent ask "..."   # use it in the agent
```

The adapter was trained on a 4-bit base; the repo serves it on the bf16 base, which usually
matches closely for LoRA classifiers. If scores look shifted, report it and serve in 4-bit.